[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/crystal_diffraction/crystal_diffraction_spacetime.ipynb)

# Crystal Deformation and Diffraction in Spacetime

The experiment of [the crystal diffraction notebook](crystal_diffraction.ipynb), a crystal layer stretched, sheared and turned, with its atoms carried by the deformation and its diffraction peaks by the deformation's adjoint, computed in the algebra of spacetime instead of the algebra of space, with the time of an observer at rest with the crystal explicit. What the atoms scatter is light, an electromagnetic field with an electric and a magnetic part for that observer. In circularly polarized light the field turns as the wave advances, its electric part into its magnetic part and back, and the phase of the wave is how far it has turned; the sense of the turn is the light's helicity. The brightness of the diffraction pattern is the energy density of the scattered field that the observer measures.

Events and wavevectors are vectors of spacetime, and the observer is its time direction `t`. The deformation acts on space and leaves `t` where it is; its adjoint under the spacetime pairing carries the reciprocal lattice, and its adjugate carries the phase fronts, the duals of the wavevectors, which in spacetime are trivectors. The incident light is the null bivector `wavevector ^ mv.x`, and the pseudoscalar `I = mv.txyz`, which squares to minus one, turns it into the same wave polarized along y, `I * incident == mv.y ^ wavevector`. Each atom's wave is the incident field turned by its phase, `(I * phases).exp() * incident`, and the intensity is the energy density `0.5 * ((field >> observer) | observer)`, positive for every field.

In the Riemann–Silberstein notation the field reads as the complex vector $\mathbf F = \mathbf E + i\mathbf B$, the turn by $I$ as multiplication by $i$, a circularly polarized wave as $\mathbf F_0\,e^{i\varphi}$, and the energy density as $\tfrac12\,\mathbf F^*\!\cdot\mathbf F = \tfrac12\left(\mathbf E^2 + \mathbf B^2\right)$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import STA
from examples.animation import save_figure
from examples.optics.crystal_diffraction import render

# The algebra of spacetime, with signature (+, -, -, -).
ga = STA
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
# The pseudoscalar squares to minus one; multiplying a field by it turns its electric part into its
# magnetic part.
I = mv.txyz                                                                         # [] Pseudoscalar
# The observer at rest with the crystal: its time direction.
observer = mv.t                                                                     # [] Vector

## 1. The lattice, the deformation and its adjoint

The atoms form a square layer, events at the observer's time zero. Its reciprocal lattice is the set of wavevectors whose phase is a whole number of turns at every atom, and these are where the diffraction peaks appear. The deformation is a map on events, `Vector <- Vector`: on space the identity plus a stretch along x and a shear, turned in the plane xy, and on the observer's time direction the identity. When the crystal deforms, its reciprocal lattice moves by the adjoint of the deformation under the spacetime pairing, the map with `deformation.adjoint()(k) | x == k | deformation(x)`. The deformed peaks are the wavevectors `k'` with `deformation.adjoint()(k') == k`, found by one solve through the adjoint.

In [ ]:
# A square layer of equal atoms; one deformation per case: none, a stretch, and a stretch, a shear and a turn.
n_shells = 3
n_orders = 3
spacing = 1.0
stretches = np.array([1.0, 1.7, 1.7])
shears = np.array([0.0, 0.0, 0.6])
angles = np.array([0.0, 0.0, np.pi / 9])

In [ ]:
# The square layer's atoms, and its reciprocal lattice: the wavevectors whose phase at every atom is a
# whole number of turns.
cells = np.arange(-n_shells, n_shells + 1)
atoms = (mv.x * cells[None, :] + mv.y * cells[:, None]).reshape(-1) * spacing       # [atoms] Vector
reflections = np.arange(-n_orders, n_orders + 1)
lattice = ((mv.x * reflections[None, :] + mv.y * reflections[:, None]).reshape(-1)
           * (2 * np.pi / spacing))                                                 # [reflections] Vector

# The directions of space square to minus one: the part of a vector along x is x times its pairing
# with the inverse of x. The observer's time direction is left where it is.
local = (Vector + (stretches - 1) * mv.x * (mv.x.inverse() | Vector)
         + shears * mv.x * (mv.y.inverse() | Vector))                               # [cases] Vector <- Vector
# With x and y both squaring to minus one, a positive angle turns x toward y.
deformation = (mv.xy * (angles / 2)).exp() >> local                                 # [cases] Vector <- Vector
positions = deformation[:, None](atoms)                                             # [cases, atoms] Vector

# The adjoint pulls a phase measurement back; solve for the measurement after deformation.
adjoint = deformation.adjoint()                                                     # [cases] Vector <- Vector
reciprocal = adjoint[:, None].solve(lattice)                                        # [cases, reflections] Vector

## 2. The same phase through incidence

In spacetime the dual of a wavevector is a trivector, its phase front: a plane of constant phase together with its history along the observer's time, which the front contains because the wavevector has no part along it. The phase at an event is the regressive product of the front with the event, `phase_fronts & event`, and incidence carries that measurement across the deformation by the adjugate, `deformation.adjugate()(fronts) & x == fronts & deformation(x)`, which needs no metric. Solving through the adjugate moves the phase fronts, and they come out as the duals of the moved wavevectors: the metric and incidence agree on every phase.

In matrix notation, with the Minkowski metric $\eta = \operatorname{diag}(1, -1, -1, -1)$, the moved wavevectors read as $\eta^{-1}F^{-\mathsf T}\eta\,\mathbf g$, with $(\eta^{-1}F^{-\mathsf T}\eta\,\mathbf g)^\mathsf{T}\eta\,(F\mathbf x) = \mathbf g^\mathsf{T}\eta\,\mathbf x$.

In [ ]:
adjugate = deformation.adjugate()                                                   # [cases] Antivector <- Antivector
phase_fronts = adjugate[:, None].solve(lattice.dual())                              # [cases, reflections] Antivector
# phase_fronts == reciprocal.dual(): the same phase read through incidence or the metric.

## 3. The scattered field and its energy density

The incident light is a plane wave running along z, the null bivector `wavevector ^ mv.x` with `wavevector = observer + mv.z`: its electric field lies along x, its magnetic field along y, and it squares to zero. Each atom scatters it with the phase of its event, the pairing of the momentum transfer with the event. The incident field turned by that phase, `(I * phases).exp() * incident`, is the same wave with its polarization turned about z by the phase: circularly polarized light, whose helicity is the sign of `I` in the exponent. A turn by `I` keeps a field's energy, so at every transfer the energy density the observer measures is the incident energy, one, times the squared length of the summed phase factors.

In [ ]:
# A square section of momentum transfers in the plane of the layer, one per pixel.
half_width = 10.0
n_pixels = 161
samples = np.linspace(-half_width, half_width, n_pixels)
transfers = mv.x * samples[None, :] + mv.y * samples[:, None]                       # [rows, columns] Vector
# Light running along z, its electric field along x.
wavevector = observer + mv.z                                                        # [] Vector
incident = wavevector ^ mv.x                                                        # [] Bivector

In [ ]:
def scattered(positions: Vector, transfers: Vector, incident: Bivector) -> Bivector:
    """The wave each atom scatters to each transfer: the incident field turned by the atom's phase."""
    # Light scattered without a change of frequency has a transfer with no part along the observer's
    # time, so the time of an atom's event does not enter its phase.
    phases = transfers[..., None] | positions[..., None, None, :]                   # [..., rows, columns, atoms] Scalar
    # I squares to minus one: its exponential turns the field by the phase, electric into magnetic.
    return (I * phases).exp() * incident                                            # [..., rows, columns, atoms] Bivector


waves = scattered(positions, transfers, incident)                                   # [cases, rows, columns, atoms] Bivector
# Equal atoms scatter equally; divided by their count, an in-phase peak carries the incident field.
field = waves.sum(axis=-1) / positions.shape[-1]                                    # [cases, rows, columns] Bivector
# The field's sandwich of the observer is twice the energy and momentum it carries for that observer,
# and its pairing with the observer twice the energy density.
energy_density = 0.5 * ((field >> observer) | observer)                             # [cases, rows, columns] Scalar
save_figure(render.spacetime_comparison(positions, reciprocal, transfers, energy_density), "crystal_diffraction_spacetime");

The top row shows the atoms of each case: the square layer, the stretched layer, and the stretched, sheared and turned layer. The bottom row shows the energy density of the summed field over the section of transfers, on a logarithmic scale, with cyan rings at the reciprocal lattice points carried by the adjoint.

In [ ]:
# checks
from examples.optics.crystal_diffraction import core

# Every wavevector meets every moved atom in the phase it met the atom at rest, read through the metric
# and through incidence alike.
moved = reciprocal[:, :, None] | positions[:, None, :]                              # [cases, reflections, atoms] Scalar
np.testing.assert_allclose((moved - (lattice[:, None] | atoms[None, :])).kernel, 0.0, atol=1e-10)
np.testing.assert_allclose(((phase_fronts[:, :, None] & positions[:, None, :]) - moved).kernel, 0.0, atol=1e-10)
np.testing.assert_allclose((phase_fronts - reciprocal.dual()).kernel, 0.0, atol=1e-11)
# The incident wave is null, and I turns its polarization from x to y.
np.testing.assert_allclose(incident.squared().kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((I * incident - (mv.y ^ wavevector)).kernel, 0.0, atol=1e-12)
# At every moved reciprocal lattice point the atoms' waves arrive in phase, with the incident energy.
peaks = scattered(positions, reciprocal[:, None, :], incident).sum(axis=-1) / positions.shape[-1]
np.testing.assert_allclose((0.5 * ((peaks >> observer) | observer)).to_array(), 1.0, atol=1e-5)
# The energy density is non-negative, and it is the intensity computed in the algebra of space.
crystals_in_space = core.square(n_shells, n_orders, spacing).carried(core.deformation(stretches, shears, angles))
intensity_in_space = crystals_in_space.intensity(core.transfer_grid(half_width, n_pixels))
assert energy_density.to_array().min() >= 0.0
np.testing.assert_allclose(energy_density.to_array(), intensity_in_space.to_array(), atol=1e-6)